# 07 · Wav-KAN como cabeça da ResNet-152 — mesmo protocolo do P1

**Sessão P3 · ambiente L4.** Mesma receita, mesmas dobras e mesmo código do `02` (entrada/saída) e do `04` (MLSD): ResNet-152 ImageNet → Dropout 0,5 → cabeça; 2 fases (15 + 45 épocas); última época; AUC por probabilidade. Só a cabeça muda para uma KAN com wavelets (Wav-KAN, Bozorgasl & Chen 2024), registrada pelo plugin `cabecas_wavkan.py`. O `protocolo_comum.py` não muda; o SHA do plugin vai à parte em cada `metricas.json`.

| id | cabeça | parâmetros (E/S) | papel |
|---|---|---|---|
| W1 | Wav-KAN chapéu mexicano `[2048,128,2]` | 1.049.860 | mesma topologia da KAN spline (d), 3,7× menor |
| W2 | Wav-KAN chapéu mexicano `[2048,32,2]` | 262.468 | mesmo orçamento da MLP-128 (b) (Δ = −62) |
| W3 | Wav-KAN DoG `[2048,128,2]` | 1.049.860 | sensibilidade à wavelet |
| W4 | Wav-KAN Morlet `[2048,128,2]` | 1.049.860 | sensibilidade à wavelet |
| W5 | Wav-KAN chapéu mexicano `[2048,480,2]` | 3.936.964 | mesmo orçamento de (c) e (d) |

**O que mudar na célula de configuração:**
* `TAREFA = "es"` (entrada/saída) ou `"mlsd"` (distância do disparo, só nas entradas; só W1 e W2).
* `ETAPA = "principal"`: W1 e W2 com as sementes 42, 43 e 44, na ordem semente 42 (W1, W2) → 43 → 44, para que um saldo curto não deixe W2 sem nenhuma semente. `ETAPA = "sensibilidade"`: W3 → W4 → W5 só com a semente 42 (só entrada/saída; por último, se sobrar saldo).
* `TESTE_RAPIDO = True` na 1ª vez de cada combinação de `TAREFA` e `ETAPA`: cada cabeça da etapa roda 2 + 2 épocas na dobra 0 e há um teste de matar e retomar (pasta de teste, ~5–8 min). Depois mude para `False` e **Executar tudo**.

**Se a sessão cair:** abra de novo e **Executar tudo**: unidades prontas são puladas e a do meio retoma da última época salva. **Uma aba, um arquivo:** não abra este notebook em duas abas; para rodar outra coisa ao mesmo tempo, use outro notebook (ou uma cópia com outro nome e outro `ABA`).

Resultados em `resultados/wavkan/` (E/S) e `resultados/wavkan_mlsd/` (MLSD): por unidade, `metricas.json`, `pred_teste.csv`, `historico.csv` e `diagnostico_wavkan.json` (scale/translation ao fim de cada fase); por experimento, `fila.json`, `runs.csv`, `resumo.json` e `oof.csv`. Se a perda virar NaN, a fila para e grava `falha_nan.json` na pasta da unidade.

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "d02f6ca6f41f0530df0f0cc3886382a8c35311ac1ec812c1f65fa74e18d800ed"   # cabecas_wavkan.py
SHA_MLSD_ESPERADO = "e2138640a83ef7587200ad6e051262b6a49e5b60027f6b4a61407fd1f61889f7"     # configs_mlsd.py (só na MLSD)
SHA_DOBRAS_ESPERADO = "ccef0fdcb8e98d3e94dbf5373eafed01936487b41eb8c41102f04970dc551f58"   # as dobras do P1
ABA = "A"                    # dono das travas desta aba
PERMITIR_RESERVA = False     # True só para rodar na T4 como reserva (marcado com † na tabela)
TAREFA = "es"                # "es" (entrada/saída) | "mlsd" (distância; só W1 e W2)
ETAPA = "principal"          # "principal" (W1, W2; sementes 42, 43, 44) | "sensibilidade" (W3, W4, W5; semente 42; só "es")
FILTRO_SEMENTES = None       # ex.: [42] para rodar só uma semente
TESTE_RAPIDO = True          # 1ª vez de cada TAREFA/ETAPA: True; depois False
CKPT_INTERVALO_MIN = None    # None = usa a recomendação do 01
DESLIGAR_AO_FIM = True       # desconecta o ambiente quando a rodada termina

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

assert TAREFA in ("es", "mlsd") and ETAPA in ("principal", "sensibilidade"), (TAREFA, ETAPA)
assert not (TAREFA == "mlsd" and ETAPA == "sensibilidade"), "Na MLSD só rodam W1 e W2 (ETAPA = \"principal\")."
pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
ARQ_PLUGIN = f"{DIR_DRIVE}/cabecas_wavkan.py"
assert pc.sha256_arquivo(ARQ_PLUGIN) == SHA_PLUGIN_ESPERADO, (
    "O cabecas_wavkan.py do Drive não é o publicado. Não rode nada; avise a sessão P3.")
pc.carregar_plugin(ARQ_PLUGIN)          # registra as cabeças e configs Wav-KAN (o SHA vai para cada metricas.json)
cw = sys.modules["cabecas_wavkan"]
cm = None
if TAREFA == "mlsd":
    ARQ_MLSD = f"{DIR_DRIVE}/configs_mlsd.py"
    assert pc.sha256_arquivo(ARQ_MLSD) == SHA_MLSD_ESPERADO, (
        "O configs_mlsd.py do Drive não é o publicado. Não rode nada; avise o P1.")
    pc.carregar_plugin(ARQ_MLSD)        # só as funções das dobras da MLSD (fold_dist_dk)
    cm = sys.modules["configs_mlsd"]
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "… | plugin", SHA_PLUGIN_ESPERADO[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

In [ ]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
assert sess.meta["dobras_sha256"] == SHA_DOBRAS_ESPERADO, "O CSV de dobras não é o das rodadas do P1. Pare e avise a sessão P3."
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))
if cm is not None:
    sess.df = cm.adicionar_colunas_dist(sess.df)       # fold_dist_d0..d4
    sess.meta["mlsd"] = "treino só nas entradas; teste = dobra inteira (saídas sem métrica)"
    print("Dobras da MLSD (teste da dobra k; as saídas nunca treinam):")
    display(cm.checar_colunas_dist(sess.df))

In [ ]:
cal = json.loads((Path(R) / "calibracao_l4.json").read_text())
PREC = cal.get("decisao_precisao")
assert PREC in ("amp", "fp32"), "Rode o 01 (ponte fp32 × AMP) antes."
CKPT = cal.get("recomendacao_ckpt_intervalo_min", 0.0) if CKPT_INTERVALO_MIN is None else CKPT_INTERVALO_MIN
T = "es" if TAREFA == "es" else "dist"
EXP = cw.EXPERIMENTO[T]
ids = list(cw.PRINCIPAIS) if ETAPA == "principal" else list(cw.SENSIBILIDADE)
sementes = [42, 43, 44] if ETAPA == "principal" else [42]
if FILTRO_SEMENTES:
    sementes = [s for s in sementes if s in FILTRO_SEMENTES]
configs = [(cw.CONFIGS_ES if T == "es" else cw.CONFIGS_MLSD)[w] for w in ids]
fila = cw.montar_fila_sementes(EXP, configs, sementes, coluna_dobra_de=cm.coluna_dobra_dist if cm else None)

arq_b = Path(R) / "benchmark_cabecas.json"
bench = json.loads(arq_b.read_text()) if arq_b.exists() else {}
PREV = (bench.get("previsao") or {}).get(T, {})
if bench and bench.get("plugin_sha256") != SHA_PLUGIN_ESPERADO:
    print("⚠️ benchmark_cabecas.json é de outra versão do plugin; a previsão é só indicativa.")

def previsao(u):
    p = PREV.get(u["config"])
    cab = f"{cw.id_da_config(u['config'])} s{u['semente']} dobra {u['dobra']}"
    if not p:
        return f"▶ {cab} (sem previsão: rode o 06) — {pc.agora_iso()}"
    return f"▶ {cab}: previsão ~{p['min_por_dobra']:.1f} min ≈ {p['uc_por_dobra']:.2f} UC — {pc.agora_iso()}"

faltam = [u for u in fila if not (cw.dir_unidade(R, u) / "metricas.json").exists()]
print(f"Experimento {EXP} ({TAREFA}, etapa {ETAPA}) | precisão: {PREC} | checkpoint a cada ≥ {CKPT} min")
print(f"{len(fila)} unidades, {len(faltam)} a fazer (ordem: semente → cabeça → dobra):")
print(pd.DataFrame(fila).assign(id=lambda d: d.config.map(cw.id_da_config)).groupby(["semente", "id"], sort=False).size().to_string())
if PREV and faltam:
    m = sum(PREV[u["config"]]["min_por_dobra"] for u in faltam if u["config"] in PREV)
    print(f"\nPrevisão do que falta: ~{m / 60:.1f} h de L4 ≈ {m / 60 * bench['previsao']['taxa_uc_h']:.1f} UC "
          "(+ ~5 min de preparo por sessão)")

## Teste rápido (cada cabeça da etapa: 2 + 2 épocas na dobra 0; matar e retomar; pasta de teste)

In [ ]:
if TESTE_RAPIDO:
    import shutil
    EXP_T = f"t07_rapido_{T}"
    for e in (EXP_T, f"{EXP_T}_ref", f"{EXP_T}_int"):
        shutil.rmtree(f"{RT}/{e}", ignore_errors=True); shutil.rmtree(f"{CT}/{e}", ignore_errors=True)
    problemas = []
    # 1) cada cabeça da etapa, dobra 0 inteira, 2 + 2 épocas
    for cfg in configs:
        u = {**next(x for x in fila if x["config"] == cfg and x["dobra"] == 0), "experimento": EXP_T}
        t0 = time.time()
        cw.executar_fila_wavkan(sess, [u], RT, CT, precisao=PREC, limite_epocas=(2, 2), espera_trava_s=0)
        d = cw.dir_unidade(RT, u)
        r = json.loads((d / "metricas.json").read_text())
        m = r["metricas"][T]
        diag = json.loads((d / "diagnostico_wavkan.json").read_text())["cabecas"][0]
        print(f"{cw.id_da_config(cfg)} {cfg}: acc {m['acc']:.3f} | AUC {m['auc']:.3f} | F1 {m['f1_macro']:.3f} | "
              f"cabeça {r['parametros']['cabecas_total']:,} parâmetros | {time.time() - t0:.0f} s | "
              f"teste {r['n_teste']} imagens ({m['n']} na métrica)")
        print("   scale/translation ao fim da F2 (camada 0):",
              {k: round(v, 4) for k, v in diag["fim_fase2"]["camada0"].items() if k in ("scale_media", "scale_dp", "translation_media", "translation_dp")})
        if r["ambiente"]["protocolo_sha256"] != SHA_PROTOCOLO_ESPERADO: problemas.append("SHA do protocolo")
        if r["plugins"].get("cabecas_wavkan", {}).get("sha256") != SHA_PLUGIN_ESPERADO: problemas.append("SHA do plugin")
        if r["dados"].get("dobras_sha256") != SHA_DOBRAS_ESPERADO: problemas.append("SHA das dobras")
        if not m["prob_continua"]: problemas.append(f"{cfg}: probabilidades não contínuas")
        if not np.isfinite(m["auc"]) or abs(m["auc"] - m["rec_macro"]) < 1e-12: problemas.append(f"{cfg}: AUC inválida ou = revocação macro")
        if not {"fim_fase1", "fim_fase2"} <= set(diag): problemas.append(f"{cfg}: diagnóstico incompleto")
        if (r["ambiente"]["gpu"] or {}).get("nome") != GPU["nome"]: problemas.append("GPU não registrada")
    # 2) matar e retomar: para no meio da F1 e no meio da F2; o resultado tem de ser idêntico
    torch.backends.cudnn.deterministic = True
    kw = dict(precisao=PREC, limite_epocas=(2, 2), limite_imagens=256, espera_trava_s=0)
    u_ref, u_int = ({**fila[0], "experimento": f"{EXP_T}_{s}"} for s in ("ref", "int"))   # 1ª cabeça, dobra 0
    cw.executar_fila_wavkan(sess, [u_ref], RT, CT, **kw)
    for parar in (1, 3):
        try:
            cw.executar_fila_wavkan(sess, [u_int], RT, CT, parar_apos_epoca=parar, **kw)
        except pc.InterrupcaoSimulada as e:
            print("interrupção simulada:", e)
    cw.executar_fila_wavkan(sess, [u_int], RT, CT, **kw)
    torch.backends.cudnn.deterministic = False
    da, db = cw.dir_unidade(RT, u_ref), cw.dir_unidade(RT, u_int)
    h1, h2 = pd.read_csv(da / "historico.csv"), pd.read_csv(db / "historico.csv")
    p1, p2 = pd.read_csv(da / "pred_teste.csv"), pd.read_csv(db / "pred_teste.csv")
    col = f"p_{T}_1"
    dif_perda, dif_prob = float(np.abs(h1["perda"] - h2["perda"]).max()), float(np.abs(p1[col] - p2[col]).max())
    print(f"retomada: máx. dif. da perda {dif_perda:.2e} | das probabilidades {dif_prob:.2e}")
    if not (dif_perda < 1e-3 and dif_prob < 1e-3): problemas.append("a retomada não reproduziu o treino")
    if (Path(CT) / u_int["experimento"]).exists() and any((Path(CT) / u_int["experimento"]).rglob("ckpt_last.pt")):
        problemas.append("checkpoint não apagado ao fim da unidade")
    # 3) nada de imagem, cache ou checkpoint nas pastas de resultados
    proib = [p for base in (Path(RT), Path(R) / EXP) if base.exists() for p in base.rglob("*")
             if p.suffix.lower() in (".npy", ".jpg", ".jpeg", ".pt")]
    if proib: problemas.append(f"arquivos proibidos nos resultados: {proib[:3]}")
    print("Problemas:", problemas or "nenhum")
    assert not problemas, problemas
    print("✅ Teste rápido OK. ➡️ Agora mude TESTE_RAPIDO = False e clique em Executar tudo.")

## Rodada

In [ ]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar a etapa."
fila_json = Path(R) / EXP / "fila.json"
antiga = json.loads(fila_json.read_text()) if fila_json.exists() else []
pc.gravar_json_atomico(antiga + [u for u in fila if u not in antiga], fila_json)
feitos = cw.executar_fila_wavkan(sess, fila, R, C, previsao=previsao, precisao=PREC, ckpt_intervalo_min=CKPT)
print("Concluídas nesta sessão:", len(feitos))

## Resultados do experimento (runs.csv, resumo.json, oof.csv)

In [ ]:
METR = [f"{T}_{m}" for m in ("acc", "prec_macro", "rec_macro", "f1_macro", "acc_balanceada", "auc")]

def agregar_experimento(experimento):
    # Grava runs.csv (1 linha = cabeça × semente × dobra), resumo.json e oof.csv do experimento.
    runs = pc.ler_resultados(R, experimento)
    if runs.empty:
        print("Nada concluído ainda em", experimento)
        return runs
    runs = runs[~runs["teste_rapido"].astype(bool)].copy()
    d = Path(R) / experimento
    runs.to_csv(d / "runs.csv", index=False)
    resumo = {}
    for (cfg, sem), g in runs.groupby(["config", "semente"]):
        resumo.setdefault(cfg, {})[f"s{sem}"] = {
            "n_dobras": int(len(g)),
            **{m: {"media": float(g[m].mean()), "dp": float(g[m].std(ddof=1)) if len(g) > 1 else None} for m in METR},
            "treino_min_por_dobra": float(g["treino_min"].mean()),
            "params_cabeca": int(g["params_cabeca"].iloc[0]),
            "params_total": int(g["params_total"].iloc[0]),
            "gpus": sorted(g["gpu"].dropna().unique().tolist()),
        }
    pc.gravar_json_atomico(resumo, d / "resumo.json")
    partes = []
    for _, r in runs.iterrows():
        p = pd.read_csv(Path(r["caminho"]) / "pred_teste.csv")
        partes.append(p.assign(config=r["config"], semente=r["semente"], dobra=r["dobra"]))
    pd.concat(partes).to_csv(d / "oof.csv", index=False)
    print(f"{experimento}: {len(runs)} unidades -> runs.csv, resumo.json, oof.csv")
    return runs

runs = agregar_experimento(EXP)
if not runs.empty:
    display(runs.assign(id=runs.config.map(cw.id_da_config)).groupby(["id", "semente"])[
        [f"{T}_acc", f"{T}_f1_macro", f"{T}_auc", "treino_min"]].agg(["mean", "std", "count"]).round(4))

⚠️ Se `DESLIGAR_AO_FIM = True`, a célula abaixo desconecta o ambiente (não gasta unidades parado). No teste rápido ela não chega a rodar: a célula da rodada para antes.

In [ ]:
if DESLIGAR_AO_FIM:
    from google.colab import runtime
    runtime.unassign()